### Setup

In [1]:
if (exists("plot", inherits = FALSE)) rm(plot)

suppressPackageStartupMessages({
  library(tidyverse)
  library(ggtext)
  library(ggh4x)
  library(ggnewscale)
  library(vegan)
  library(scales)
  library(cowplot)
  library(readr)
  library(stringr)
})

if (requireNamespace("conflicted", quietly = TRUE)) {
  conflicted::conflict_prefer("select", "dplyr", quiet = TRUE)
  conflicted::conflict_prefer("filter", "dplyr", quiet = TRUE)
  conflicted::conflict_prefer("lag",    "dplyr", quiet = TRUE)
  conflicted::conflict_prefer("rename", "dplyr", quiet = TRUE)
}

data_dir <- "../data"
fig_dir  <- "../figs"

pdf_device <- if (capabilities("cairo")) cairo_pdf else "pdf"

### Global colours, palettes, themes, helpers

In [2]:
## Ampicillin & biomass colours
AMP_COL    <- "#6D2682"
GROWTH_COL <- "#2BBFD9"

ampicillin_col      <- AMP_COL
growth_capacity_col <- GROWTH_COL

## Pre-pulse AB palette (alpha applied later)
ab_cols <- c(
  "0"   = "#BBBBBC",
  "5"   = "#D3C2E8",
  "50"  = "#AD8CB9",
  "500" = "#6D2682"
)

## Species colours (as provided)
species_colors <- c(
  "HAMBI_1988" = "#EA7F40",
  "HAMBI_3031" = "#EAB0F6",
  "HAMBI_1896" = "#834F16",
  "HAMBI_3237" = "#1649F7",
  "HAMBI_0105" = "#FDD21C",
  "HAMBI_2443" = "#8200AA",
  "HAMBI_1842" = "#0DF8EB",
  "HAMBI_0262" = "#FD0D7A",
  "HAMBI_2160" = "#6A9940",
  "HAMBI_0403" = "#82BAFE",
  "HAMBI_2164" = "#FA2AD9",
  "HAMBI_2494" = "#FD83B5",
  "HAMBI_2159" = "#FED8BE",
  "HAMBI_0097" = "#2AF622",
  "HAMBI_2792" = "#BEDDE4",
  "HAMBI_1292" = "#C4EF35",
  "HAMBI_1279" = "#42F19F",
  "HAMBI_1299" = "#DC16FC",
  "HAMBI_1287" = "#70566A",
  "HAMBI_1972" = "#F6533B",
  "HAMBI_1977" = "#164F85",
  "HAMBI_0006" = "#22917A",
  "HAMBI_2659" = "#BA3249"
)

species_labels_html <- c(
  "HAMBI_0006" = "<i>Pseudomonas putida</i> HAMBI 6",
  "HAMBI_0097" = "<i>Acinetobacter lwoffii</i> HAMBI 97",
  "HAMBI_0105" = "<i>Agrobacterium tumefaciens</i> HAMBI 105",
  "HAMBI_0262" = "<i>Brevundimonas bullata</i> HAMBI 262",
  "HAMBI_0403" = "<i>Comamonas testosteroni</i> HAMBI 403",
  "HAMBI_1279" = "<i>Hafnia alvei</i> HAMBI 1279",
  "HAMBI_1287" = "<i>Citrobacter koseri</i> HAMBI 1287",
  "HAMBI_1292" = "<i>Morganella morganii</i> HAMBI 1292",
  "HAMBI_1299" = "<i>Kluyvera intermedia</i> HAMBI 1299",
  "HAMBI_1842" = "<i>Sphingobium yanoikuyae</i> HAMBI 1842",
  "HAMBI_1896" = "<i>Sphingobacterium spiritivorum</i> HAMBI 1896",
  "HAMBI_1972" = "<i>Aeromonas caviae</i> HAMBI 1972",
  "HAMBI_1977" = "<i>Pseudomonas chlororaphis</i> HAMBI 1977",
  "HAMBI_1988" = "<i>Chitinophaga sancti</i> HAMBI 1988",
  "HAMBI_2159" = "<i>Paraburkholderia caryophylli</i> HAMBI 2159",
  "HAMBI_2160" = "<i>Bordetella avium</i> HAMBI 2160",
  "HAMBI_2164" = "<i>Cupriavidus oxalaticus</i> HAMBI 2164",
  "HAMBI_2443" = "<i>Paracoccus denitrificans</i> HAMBI 2443",
  "HAMBI_2494" = "<i>Paraburkholderia kururiensis</i> HAMBI 2494",
  "HAMBI_2659" = "<i>Stenotrophomonas maltophilia</i> HAMBI 2659",
  "HAMBI_2792" = "<i>Moraxella canis</i> HAMBI 2792",
  "HAMBI_3031" = "<i>Niabella yanshanensis</i> HAMBI 3031",
  "HAMBI_3237" = "<i>Microvirga lotononidis</i> HAMBI 3237"
)

apex_theme <- function(base_size = 10,
                       facet_y_pt = 7.6,
                       facet_x_pt = 8.8,
                       axis_text_pt = 8.6) {
  theme_classic(base_size = base_size) +
    theme(
      panel.background   = element_rect(fill = "white", color = NA),
      plot.background    = element_rect(fill = "white", color = NA),
      panel.grid         = element_blank(),
      axis.line.x        = element_line(color = "black", linewidth = 0.4),
      axis.line.y        = element_line(color = "black", linewidth = 0.4),
      axis.text          = element_text(size = axis_text_pt),
      strip.background   = element_rect(fill = "white", color = NA),
      strip.text.x       = ggtext::element_markdown(face = "bold", size = facet_x_pt, lineheight = 1.05),
      strip.text.y       = ggtext::element_markdown(face = "bold", size = facet_y_pt, lineheight = 1.05),
      strip.text.y.left  = ggtext::element_markdown(face = "bold", size = facet_y_pt, lineheight = 1.05),
      legend.background  = element_rect(fill = "white", colour = NA),
      legend.key         = element_rect(fill = "white", colour = NA),
      legend.text        = ggtext::element_markdown(),
      legend.title       = ggtext::element_markdown(face = "bold"),
      legend.position    = "bottom"
    )
}

comp_theme <- function() {
  theme_classic(base_size = 10) +
    theme(
      panel.background   = element_rect(fill = "white", color = NA),
      plot.background    = element_rect(fill = "white", color = NA),
      axis.line          = element_line(color = "black", linewidth = 0.4),
      axis.text.x        = element_text(size = 6.4),
      axis.text.y        = element_text(size = 8.4),
      strip.background   = element_rect(fill = "white", color = NA),
      strip.text.x       = ggtext::element_markdown(face = "bold", size = 8.2),
      strip.text.y       = ggtext::element_markdown(face = "bold", size = 7.4),
      strip.text.y.left  = ggtext::element_markdown(face = "bold", size = 7.4),
      legend.key.height  = unit(6, "pt"),
      legend.key.width   = unit(10, "pt"),
      legend.margin      = margin(t = 2, r = 2, b = 2, l = 2),
      legend.text        = ggtext::element_markdown(size = 7.8, lineheight = 1.0),
      legend.background  = element_rect(fill = "white", colour = NA),
      legend.position    = "bottom"
    )
}

fix_names <- function(x) {
  nms <- names(x)
  bad <- is.na(nms) | nms == ""
  if (any(bad)) nms[bad] <- paste0("X", seq_len(sum(bad)))
  names(x) <- make.unique(nms)
  x
}

## shading alphas
shade_alpha_comp  <- 0.18
shade_alpha_other <- 0.16

### Load community composition data + metadata

In [3]:
species_raw <- fix_names(
  read.delim(file.path(data_dir, "species_abundance.txt"),
             check.names = FALSE, stringsAsFactors = FALSE)
)

species <- species_raw %>%
  mutate(
    Day = case_when(
      grepl("d4",  sample) ~ 4,
      grepl("d8",  sample) ~ 8,
      grepl("d12", sample) ~ 12,
      grepl("d16", sample) ~ 16,
      grepl("d20", sample) ~ 20,
      TRUE ~ NA_real_
    ),
    Evo_orig = case_when(
      grepl("^a_", sample) ~ "All ancestral",
      grepl("^d_", sample) ~ "Evolved 1972",
      grepl("^e_", sample) ~ "Evolved 1977",
      grepl("^f_", sample) ~ "All evolved",
      TRUE ~ NA_character_
    ),
    Replicate = case_when(
      grepl("_1_", sample) ~ 1L,
      grepl("_2_", sample) ~ 2L,
      grepl("_3_", sample) ~ 3L,
      grepl("_4_", sample) ~ 4L,
      TRUE ~ NA_integer_
    ),
    AB = case_when(
      grepl("_t1_", sample) ~ 5L,
      grepl("_t2_", sample) ~ 50L,
      grepl("_t3_", sample) ~ 500L,
      grepl("_t4_", sample) ~ 0L,
      TRUE ~ NA_integer_
    )
  ) %>%
  filter(!is.na(Day), !is.na(Evo_orig), !is.na(AB), !is.na(Replicate)) %>%
  mutate(
    Evo_html = factor(
      Evo_orig,
      levels = c("All ancestral", "Evolved 1972", "Evolved 1977", "All evolved"),
      labels = c(
        "Ancestral<br>community",
        "Dominant species<br>(<i>Aeromonas</i><br><i>caviae</i>)<br>primed",
        "Subdominant species<br>(<i>Pseudomonas</i><br><i>chlororaphis</i>)<br>primed",
        "Full primed<br>community"
      )
    ),
    AB_fac   = factor(AB, levels = c(0, 5, 50, 500), labels = c("0", "5", "50", "500")),
    AB_super = factor("Pre-pulse priming ampicillin concentration (\u00B5g ml\u207B\u00B9)"),
    Replicate = factor(Replicate),
    strainID = gsub("-", "_", strainID)
  )

meta <- species %>%
  select(sample, Day, Evo_html, Evo_orig, AB_fac, AB, Replicate, AB_super) %>%
  distinct()

all_days <- sort(unique(species$Day))

### ED Fig 1: Cell density over time

In [4]:
cell_data_file <- file.path(data_dir, "cell_density_ATP.txt")

df_raw <- read.table(cell_data_file, header = TRUE, check.names = FALSE, sep = "", dec = ",")

df_cell <- df_raw %>%
  mutate(
    sample = sample_id,
    Evo = case_when(
      grepl("^a_", sample_id) ~ "All ancestral",
      grepl("^d_", sample_id) ~ "Evolved 1972",
      grepl("^e_", sample_id) ~ "Evolved 1977",
      grepl("^f_", sample_id) ~ "All evolved",
      TRUE ~ NA_character_
    ),
    Replicate = case_when(
      grepl("_1_", sample_id) ~ 1L,
      grepl("_2_", sample_id) ~ 2L,
      grepl("_3_", sample_id) ~ 3L,
      grepl("_4_", sample_id) ~ 4L,
      TRUE ~ NA_integer_
    ),
    Primer_pulse = case_when(
      grepl("_t1_", sample_id) ~ 5L,
      grepl("_t2_", sample_id) ~ 50L,
      grepl("_t3_", sample_id) ~ 500L,
      grepl("_t4_", sample_id) ~ 0L,
      TRUE ~ NA_integer_
    )
  ) %>%
  filter(
    Evo %in% c("All ancestral", "Evolved 1977", "Evolved 1972", "All evolved"),
    !is.na(Replicate), !is.na(Primer_pulse)
  ) %>%
  mutate(
    Evo_html = factor(
      Evo,
      levels = c("All ancestral", "Evolved 1972", "Evolved 1977", "All evolved"),
      labels = c(
        "Ancestral<br>community",
        "Dominant species<br>(<i>Aeromonas</i><br><i>caviae</i>)<br>primed",
        "Subdominant species<br>(<i>Pseudomonas</i><br><i>chlororaphis</i>)<br>primed",
        "Full primed<br>community"
      )
    ),
    AB_fac   = factor(Primer_pulse, levels = c(0, 5, 50, 500), labels = c("0", "5", "50", "500")),
    AB_super = factor("Pre-pulse priming ampicillin concentration (\u00B5g ml\u207B\u00B9)"),
    Replicate = factor(Replicate)
  ) %>%
  filter(is.finite(cell_number_ml)) %>%
  droplevels()

shading_meta_cell <- df_cell %>% distinct(Evo_html, AB_fac, AB_super)

phase_rects_cell <- shading_meta_cell %>%
  crossing(phase = c("pre", "main")) %>%
  mutate(
    xmin = ifelse(phase == "pre", 4, 12),
    xmax = ifelse(phase == "pre", 8, 16),
    ymin = -Inf, ymax = Inf,
    base_col = ifelse(phase == "pre", ab_cols[as.character(AB_fac)], ab_cols["500"]),
    fill = alpha(base_col, shade_alpha_other)
  )

phase_bounds_cell <- tibble(x = c(4, 8, 12, 16), y = -Inf, yend = Inf)

p_celldens_over_time <- ggplot(df_cell, aes(x = day)) +
  geom_rect(
    data = phase_rects_cell,
    aes(xmin = xmin, xmax = xmax, ymin = ymin, ymax = ymax, fill = fill),
    inherit.aes = FALSE, colour = NA
  ) +
  scale_fill_identity(guide = "none") +
  geom_segment(
    data = phase_bounds_cell,
    aes(x = x, xend = x, y = y, yend = yend),
    inherit.aes = FALSE, linetype = "dashed", linewidth = 0.35,
    colour = alpha("black", 0.35)
  ) +
  stat_summary(aes(y = cell_number_ml), fun = mean, geom = "line",
               linewidth = 0.9, colour = growth_capacity_col) +
  stat_summary(aes(y = cell_number_ml), fun.data = "mean_cl_boot",
               linewidth = 0.7, colour = growth_capacity_col) +
  geom_point(aes(y = cell_number_ml), alpha = 0.28, size = 1.7,
             colour = growth_capacity_col,
             position = position_jitter(width = 0.15, height = 0)) +
  ggh4x::facet_nested(
    rows = vars(Evo_html),
    cols = vars(AB_super, AB_fac),
    labeller = labeller(AB_super = label_value),
    drop = TRUE
  ) +
  scale_x_continuous(breaks = c(0, 4, 8, 12, 16, 20), limits = c(0, 20)) +
  labs(x = "Day", y = "Cells ml\u207B\u00B9") +
  apex_theme(base_size = 10, facet_y_pt = 7.6, facet_x_pt = 8.8, axis_text_pt = 8.6)

ggsave(file.path(fig_dir, "ED_Fig1.pdf"),
       p_celldens_over_time, width = 8.27, height = 8.8, device = pdf_device)

Warning message:
“Removed 16 rows containing missing values or values outside the scale range
(`geom_point()`).”


### ED Fig 2: P. chlororaphis extinction + stats

In [5]:
## ED FIGURE 2 — P. chlororaphis extinction vs background pre-exposure
pc_id <- "HAMBI_1977"

evo_levels <- c("All ancestral","Evolved 1972","Evolved 1977","All evolved")
evo_labels_html <- c(
  "Ancestral<br>community",
  "Dominant species<br>(<i>Aeromonas</i><br><i>caviae</i>)<br>primed",
  "Subdominant species<br>(<i>Pseudomonas</i><br><i>chlororaphis</i>)<br>primed",
  "Full primed<br>community"
)

species_ext <- species %>%
  mutate(strainID = gsub("-", "_", strainID))

pchl_by_sample <- species_ext %>%
  filter(strainID == pc_id) %>%
  group_by(sample, Evo_orig) %>%
  summarise(abund = sum(f_correct, na.rm = TRUE), .groups = "drop") %>%
  mutate(
    extinct  = as.integer(abund <= 0),
    Evo_html = factor(Evo_orig, levels = evo_levels, labels = evo_labels_html)
  )

pchl_counts <- pchl_by_sample %>%
  group_by(Evo_orig, Evo_html) %>%
  summarise(
    n_extinct = sum(extinct, na.rm = TRUE),
    n_total   = n(),
    prop_ext  = n_extinct / pmax(n_total, 1L),
    .groups   = "drop"
  ) %>%
  rowwise() %>%
  mutate(
    ci_low  = binom.test(n_extinct, n_total)$conf.int[1],
    ci_high = binom.test(n_extinct, n_total)$conf.int[2]
  ) %>%
  ungroup()

## ---- STATS: print to screen only (no files) ----
glm_fit <- tryCatch(
  glm(cbind(n_extinct, n_total - n_extinct) ~ Evo_orig,
      data = pchl_counts, family = binomial),
  error = function(e) NULL
)

cat("\n===== ED Fig 2 stats: P. chlororaphis extinction vs Evo_orig =====\n")
print(pchl_counts)

if (!is.null(glm_fit) && isTRUE(glm_fit$converged)) {
  cat("\n--- Binomial GLM summary ---\n")
  print(summary(glm_fit))
  cat("\n--- Likelihood ratio test (Chisq) ---\n")
  print(anova(glm_fit, test = "Chisq"))
} else {
  cat("\n--- GLM failed/non-converged; using chi-square on contingency table ---\n")
  tab <- pchl_counts %>%
    transmute(Evo_orig, extinct = n_extinct, present = n_total - n_extinct) %>%
    column_to_rownames("Evo_orig") %>%
    as.matrix()
  chi_res <- suppressWarnings(chisq.test(tab, simulate.p.value = TRUE, B = 1e5))
  print(chi_res)
}

## ---- FIGURE: save ----
p_pchl <- ggplot(pchl_counts, aes(x = Evo_html, y = prop_ext)) +
  geom_pointrange(aes(ymin = ci_low, ymax = ci_high), linewidth = 0.6) +
  scale_y_continuous(
    expression(paste("Extinction probability (", italic("P. chlororaphis"), ")")),
    limits = c(0, 1),
    labels = scales::percent_format(accuracy = 1)
  ) +
  labs(x = NULL) +
  apex_theme(base_size = 10) +
  theme(axis.text.x = ggtext::element_markdown(angle = 0, hjust = 0.5))

ggsave(file.path(fig_dir, "ED_Fig2.pdf"),
       p_pchl, width = 6.2, height = 3.8, units = "in", device = pdf_device)


===== ED Fig 2 stats: P. chlororaphis extinction vs Evo_orig =====
# A tibble: 4 × 7
  Evo_orig      Evo_html               n_extinct n_total prop_ext ci_low ci_high
  <chr>         <fct>                      <int>   <int>    <dbl>  <dbl>   <dbl>
1 All ancestral Ancestral<br>community         0      80    0      0      0.0451
2 All evolved   Full primed<br>commun…        42      80    0.525  0.410  0.638 
3 Evolved 1972  Dominant species<br>(…         0      80    0      0      0.0451
4 Evolved 1977  Subdominant species<b…        59      80    0.738  0.627  0.830 

--- Binomial GLM summary ---

Call:
glm(formula = cbind(n_extinct, n_total - n_extinct) ~ Evo_orig, 
    family = binomial, data = pchl_counts)

Coefficients:
                       Estimate Std. Error z value Pr(>|z|)
(Intercept)          -2.709e+01  5.177e+04  -0.001        1
Evo_origAll evolved   2.719e+01  5.177e+04   0.001        1
Evo_origEvolved 1972 -2.622e-09  7.322e+04   0.000        1
Evo_origEvolved 1977  2.812e

### ED Fig 3: Richness over time

In [6]:
species_mat_df <- species %>%
  group_by(sample, strainID) %>%
  summarise(f = sum(f_correct, na.rm = TRUE), .groups = "drop") %>%
  tidyr::pivot_wider(names_from = strainID, values_from = f, values_fill = 0)

species_mat <- as.matrix(species_mat_df[, -1])
rownames(species_mat) <- species_mat_df$sample

alpha_df <- tibble(
  sample   = rownames(species_mat),
  Richness = rowSums(species_mat > 0)
) %>%
  left_join(meta, by = "sample")

shading_meta_alpha <- alpha_df %>% distinct(Evo_html, AB_fac, AB_super)

phase_rects_alpha <- shading_meta_alpha %>%
  crossing(phase = c("pre", "main")) %>%
  mutate(
    xmin = ifelse(phase == "pre", 4, 12),
    xmax = ifelse(phase == "pre", 8, 16),
    ymin = -Inf, ymax = Inf,
    base_col = ifelse(phase == "pre", ab_cols[as.character(AB_fac)], ab_cols["500"]),
    fill = alpha(base_col, shade_alpha_other)
  )

phase_bounds_alpha <- tibble(x = c(4, 8, 12, 16), y = -Inf, yend = Inf)

p_rich <- ggplot(alpha_df, aes(x = Day, y = Richness)) +
  geom_rect(
    data = phase_rects_alpha,
    aes(xmin = xmin, xmax = xmax, ymin = ymin, ymax = ymax, fill = fill),
    inherit.aes = FALSE, colour = NA
  ) +
  scale_fill_identity(guide = "none") +
  geom_segment(
    data = phase_bounds_alpha,
    aes(x = x, xend = x, y = y, yend = yend),
    inherit.aes = FALSE, linetype = "dashed", linewidth = 0.35,
    colour = alpha("black", 0.35)
  ) +
  stat_summary(aes(group = 1), fun = mean, geom = "line",
               linewidth = 0.9, colour = growth_capacity_col) +
  stat_summary(aes(group = 1), fun.data = "mean_cl_boot",
               geom = "errorbar", width = 0.35, linewidth = 0.7,
               colour = growth_capacity_col) +
  geom_point(alpha = 0.28, size = 1.7, colour = growth_capacity_col,
             position = position_jitter(width = 0.15, height = 0)) +
  ggh4x::facet_nested(
    rows = vars(Evo_html),
    cols = vars(AB_super, AB_fac),
    labeller = labeller(AB_super = label_value)
  ) +
  scale_x_continuous(breaks = c(0, 4, 8, 12, 16, 20), limits = c(0, 20)) +
  labs(x = "Day", y = "Richness (no. of species)") +
  apex_theme(base_size = 10)

ggsave(file.path(fig_dir, "ED_Fig3.pdf"),
       p_rich, width = 8.27, height = 8.8, units = "in", device = pdf_device)

Warning message:
“Removed 48 rows containing missing values or values outside the scale range
(`geom_point()`).”


### ED Fig 4: AMP timecourse

In [7]:
#log scale

library(dplyr)
library(tidyr)
library(tibble)
library(ggplot2)
library(scales)
library(ggh4x)

amp_df <- read.table(
  file.path(data_dir, "AMP_conc.txt"),
  header = TRUE, sep = "\t", dec = ",",
  check.names = FALSE, stringsAsFactors = FALSE
)

amp_df <- amp_df %>%
  filter(Evo != "No_BAC") %>%
  mutate(
    AMP_conc = suppressWarnings(as.numeric(AMP_conc)),
    Day      = as.integer(Day),
    Evo = factor(
      Evo,
      levels = c("ANC", "E1972", "E1977", "EVO"),
      labels = c(
        "Ancestral<br>community",
        "Dominant species<br>(<i>Aeromonas</i><br><i>caviae</i>)<br>primed",
        "Subdominant species<br>(<i>Pseudomonas</i><br><i>chlororaphis</i>)<br>primed",
        "Full primed<br>community"
      )
    ),
    AB_fac = factor(
      AB,
      levels = c(0, 5, 50, 500),
      labels = c("0", "5", "50", "500")
    ),
    AB_super = factor("Pre-pulse priming ampicillin concentration (\u00B5g ml\u207B\u00B9)")
  ) %>%
  filter(is.finite(AMP_conc), AMP_conc >= 0.1) %>%
  droplevels()

shading_meta_amp <- amp_df %>%
  distinct(Evo, AB_fac, AB_super)

phase_rects_amp <- shading_meta_amp %>%
  crossing(phase = c("pre", "main")) %>%
  mutate(
    xmin = ifelse(phase == "pre", 4, 12),
    xmax = ifelse(phase == "pre", 8, 16),
    ymin = 0.1,
    ymax = 500,
    base_col = ifelse(
      phase == "pre",
      ab_cols[as.character(AB_fac)],
      ab_cols["500"]
    ),
    fill = alpha(base_col, shade_alpha_other)
  )

phase_bounds_amp <- tibble(
  x = c(4, 8, 12, 16),
  y = 0.1,
  yend = 500
)

# Custom log ticks for leftmost facet column only
# Because x expansion is removed below, x = 0 is exactly the panel border
log_ticks_amp <- tibble(
  y = c(
    0.1,
    0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9,
    1,
    2, 3, 4, 5, 6, 7, 8, 9,
    10,
    20, 30, 40, 50, 60, 70, 80, 90,
    100,
    200, 300, 400, 500
  )
) %>%
  mutate(
    tick_type = case_when(
      y %in% c(0.1, 1, 10, 100) ~ "major",
      y %in% c(0.5, 5, 50, 500) ~ "mid",
      TRUE ~ "minor"
    ),
    x = 0,
    xend = case_when(
      tick_type == "major" ~ 1.30,
      tick_type == "mid"   ~ 0.95,
      TRUE                 ~ 0.65
    ),
    lw = case_when(
      tick_type == "major" ~ 0.45,
      tick_type == "mid"   ~ 0.35,
      TRUE                 ~ 0.28
    )
  ) %>%
  crossing(
    Evo = levels(amp_df$Evo),
    AB_fac = factor("0", levels = levels(amp_df$AB_fac)),
    AB_super = levels(amp_df$AB_super)
  )

p_20d <- ggplot(amp_df, aes(x = Day, y = AMP_conc)) +
  geom_rect(
    data = phase_rects_amp,
    aes(xmin = xmin, xmax = xmax, ymin = ymin, ymax = ymax, fill = fill),
    inherit.aes = FALSE,
    colour = NA
  ) +
  scale_fill_identity(guide = "none") +
  geom_segment(
    data = phase_bounds_amp,
    aes(x = x, xend = x, y = y, yend = yend),
    inherit.aes = FALSE,
    linetype = "dashed",
    linewidth = 0.35,
    colour = alpha("black", 0.35)
  ) +
  geom_segment(
    data = log_ticks_amp,
    aes(x = x, xend = xend, y = y, yend = y, linewidth = lw),
    inherit.aes = FALSE,
    colour = "black",
    lineend = "butt"
  ) +
  scale_linewidth_identity() +
  stat_summary(
    fun = mean,
    geom = "line",
    linewidth = 0.9,
    colour = ampicillin_col
  ) +
  stat_summary(
    fun.data = "mean_cl_boot",
    geom = "errorbar",
    width = 0.35,
    linewidth = 0.7,
    colour = ampicillin_col
  ) +
  geom_point(
    alpha = 0.30,
    size = 1.8,
    position = position_jitter(width = 0.10, height = 0),
    colour = "black"
  ) +
  ggh4x::facet_nested(
    rows = vars(Evo),
    cols = vars(AB_super, AB_fac),
    labeller = labeller(AB_super = label_value)
  ) +
  scale_x_continuous(
    breaks = c(0, 4, 8, 12, 16, 20),
    limits = c(0, 20.6),
    expand = c(0, 0)
  ) +
  scale_y_log10(
    limits = c(0.1, 500),
    breaks = c(0.1, 1, 10, 100, 500),
    labels = c("0.1", "1", "10", "100", "500"),
    expand = c(0, 0)
  ) +
  labs(
    x = "Day",
    y = "Ampicillin concentration (log10 scale, \u00B5g ml\u207B\u00B9)"
  ) +
  coord_cartesian(clip = "off") +
  apex_theme(base_size = 10) +
  theme(
    panel.grid.major.y = element_blank(),
    panel.grid.minor.y = element_blank()
  )

ggsave(
  file.path(fig_dir, "ED_Fig4.pdf"),
  p_20d,
  width = 8.27,
  height = 8.8,
  units = "in",
  device = pdf_device
)